# Backtest Visualization (Updated)
This notebook visualizes the trading performance of a trained agent on stock data, calculating real PnL using Parquet Close prices and accounting for transaction costs.

In [30]:
import os
import torch as th
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from elegantrl.agents.AgentPPO import AgentDiscretePPO
# Note: Adjusted import path based on project structure
from elegantrl.envs.SingleStockTradingEnv import SingleStockTradingEnv

In [31]:
# --- CONFIGURATION ---
MODEL_PATH = "./SingleStockTradingEnv-v3_DiscretePPO_0/act.pth"
OHLCV_DIR = "/opt/rws/repos/RWS_LightGBM/data/reprocessed/stocks_m5_mdv_gt_100M"
DATA_PATH = "./data"
NUM_PLOTS = 10
NET_DIMS = [256, 128]
COST_PCT = 1e-4  # 1 basis point transaction cost

In [32]:
# --- HELPERS ---
def load_trained_agent(model_path, state_dim, action_dim, net_dims):
    """Initializes agent with required arguments and loads weights."""
    # Force CPU to avoid device mismatch during visualization
    agent = AgentDiscretePPO(net_dims=net_dims, state_dim=state_dim, action_dim=action_dim, gpu_id=-1)
    
    print(f"| Loading weights from {model_path}")
    # weights_only=False for PyTorch 2.6 compatibility with custom classes
    loaded_obj = th.load(model_path, map_location=th.device('cpu'), weights_only=False)
    
    # Support both state_dict and full model objects
    if isinstance(loaded_obj, dict):
        agent.act.load_state_dict(loaded_obj)
    elif isinstance(loaded_obj, th.nn.Module):
        agent.act.load_state_dict(loaded_obj.state_dict())
    else:
        raise TypeError(f"Unexpected type for loaded model: {type(loaded_obj)}")
    
    agent.act.eval()
    return agent

def get_ohlcv_context_slice(ticker, timestamps, ohlcv_dir, padding_days=1):
    """
    Fetches the trading day plus surrounding context.
    Each trading day is approx 78 bars (5m).
    """
    path = os.path.join(ohlcv_dir, f"{ticker}.parquet")
    if not os.path.exists(path):
        print(f"| Warning: OHLCV file for {ticker} not found at {path}")
        return None, None
    
    full_df = pd.read_parquet(path).sort_values('timestamp').reset_index(drop=True)
    target_times = pd.to_datetime(timestamps, unit='ns') 
    
    # Find the start and end indices of the actual trading day
    day_indices = full_df[full_df['timestamp'].isin(target_times)].index
    if len(day_indices) == 0:
        return None, None
        
    start_idx = day_indices.min()
    end_idx = day_indices.max()
    
    # Calculate padding (approx 78 bars per day)
    pad = 78 * padding_days
    context_start = max(0, start_idx - pad)
    context_end = min(len(full_df) - 1, end_idx + pad)
    
    # Slice the context
    context_df = full_df.iloc[context_start : context_end + 1].copy()
    
    # Provide the relative offset so the markers know where the trading day starts
    trade_day_offset = start_idx - context_start
    
    return context_df, trade_day_offset

def plot_episode_with_context(context_df, trade_day_offset, actions, ticker, ep_idx):
    """
    Plots the full context but aligns actions only to the trading window.
    """
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        row_heights=[0.7, 0.3], vertical_spacing=0.05)

    # 1. Initialize PnL tracking for the WHOLE context (mostly zeros except trading day)
    real_pnl_history = np.zeros(len(context_df))
    entry_price = 0.0
    trade_status = 0 

    # 2. Map actions to the specific window within the context
    # actions list corresponds to the trading day slice starting at trade_day_offset
    for i in range(len(actions)):
        context_idx = trade_day_offset + i
        current_close = context_df['close'].iloc[context_idx]
        
        if trade_status == 0:
            if actions[i] == 1: # OPEN LONG
                entry_price = current_close * (1 + COST_PCT)
                trade_status = 1
            elif actions[i] == 2: # OPEN SHORT
                entry_price = current_close * (1 - COST_PCT)
                trade_status = 2
        elif actions[i] != 0 or i == (len(actions) - 1):
            pnl = (current_close / entry_price) - 1.0 if trade_status == 1 else 1.0 - (current_close / entry_price)
            real_pnl_history[context_idx] = pnl
            trade_status = 0

    # 3. Candlesticks (Full Context)
    fig.add_trace(go.Candlestick(
        x=context_df['timestamp'], open=context_df['open'], high=context_df['high'],
        low=context_df['low'], close=context_df['close'], name='Context Price'
    ), row=1, col=1)

    # 4. Highlight the actual Trading Day window
    trade_start_time = context_df['timestamp'].iloc[trade_day_offset]
    trade_end_time = context_df['timestamp'].iloc[trade_day_offset + len(actions) - 1]
    
    fig.add_vrect(
        x0=trade_start_time, x1=trade_end_time,
        fillcolor="white", opacity=0.1, line_width=0,
        annotation_text="Trading Window", annotation_position="top left",
        row=1, col=1
    )

    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "mon"]), # Hide weekends
            dict(bounds=[16, 9.5], pattern="hour"), # Hide 4:00 PM to 9:30 AM
        ],
        row=1, col=1
    )
    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "mon"]),
            dict(bounds=[16, 9.5], pattern="hour"),
        ],
        row=2, col=1
    )
    
    fig.update_layout(
        title=f"Context Backtest: {ticker} (Episode {ep_idx})", 
        xaxis_rangeslider_visible=False, 
        template='plotly_dark'
    )

    # 5. Trade Markers (Only in Trading Window)
    for i, action in enumerate(actions):
        context_idx = trade_day_offset + i
        t = context_df['timestamp'].iloc[context_idx]
        if action == 1: # LONG
            fig.add_annotation(x=t, y=context_df['low'].iloc[context_idx], text="▲", font=dict(color="green", size=18), showarrow=False)
        elif action == 2: # SHORT
            fig.add_annotation(x=t, y=context_df['high'].iloc[context_idx], text="▼", font=dict(color="red", size=18), showarrow=False)
        elif i > 0 and action == 0 and actions[i-1] != 0:
            fig.add_annotation(x=t, y=context_df['close'].iloc[context_idx], text="✘", font=dict(color="cyan", size=14), showarrow=False)

    # 6. Cumulative PnL (Aligned to context)
    fig.add_trace(go.Scatter(x=context_df['timestamp'], y=np.cumsum(real_pnl_history), 
                             name='Realized PnL', fill='tozeroy', line=dict(color='#00ffcc')), row=2, col=1)

    fig.update_layout(title=f"Context Backtest: {ticker} (Episode {ep_idx})", 
                      xaxis_rangeslider_visible=False, template='plotly_dark')
    fig.show()

In [33]:
def run_analysis():
    env_args = {'data_path': DATA_PATH, 'episode_days': 1, 'price_column': 'target_reg_5m_logret', 'if_day_trade': True, 'cost_pct': COST_PCT}
    env = SingleStockTradingEnv(**env_args)
    agent = load_trained_agent(MODEL_PATH, env.state_dim, env.action_dim, NET_DIMS)
    
    for i in range(NUM_PLOTS):
        state, _ = env.reset()
        ticker, times = env.current_ticker, env.current_time_slice
        
        actions = []
        done = False
        while not done:
            s_tensor = th.as_tensor(state[np.newaxis], dtype=th.float32)
            action = agent.act(s_tensor).detach().cpu().numpy()[0]
            state, reward, done, _, _ = env.step(action)
            actions.append(action)
            
        context_df, trade_day_offset = get_ohlcv_context_slice(ticker, times, OHLCV_DIR, padding_days=1)
        if context_df is not None:
            plot_episode_with_context(context_df, trade_day_offset, actions, ticker, i)

In [34]:
run_analysis()

| Environment loaded with 310 tickers.
| Input Features: 85 (Targets Removed)
| Loading weights from ./SingleStockTradingEnv-v3_DiscretePPO_0/act.pth
